# 📊 Rosetta — inventário completo

Varre a `DDDDLSRC` inteira e responde duas perguntas diferentes:

1. **Pré-seleção estrutural** (Motor 01): o fonte fechou? o tipo é suportado? as
   dependências estão limpas? → produz `APTAS`
2. **Certeza real**: rodando parser + gerador de fato em cada apta, quais geram SQL
   **sem nenhum aviso**? → produz `GARANTIDAS`

A diferença importa: `APTA` é candidata, `GARANTIDA` é resultado verificado.

⏱️ Leva alguns minutos. Não precisa rodar toda vez — o notebook `01_gerar_view` funciona
sozinho para uma view específica.

---
### 🔒 Somente leitura — nenhuma consulta de escrita passa por `rosetta.seguranca`.

## 1. 🎛️ Widgets e carga do pacote

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("catalog_raw",    "platform_dev",       "10 | Catalog origem")
dbutils.widgets.text("schema_raw",     "sap_s4_nc2_raw",     "11 | Schema origem (raw)")
dbutils.widgets.text("tabela_ddl",     "tab_ddddlsrc",       "12 | Tabela DDDDLSRC")
dbutils.widgets.text("tabela_dep",     "ddldependency",      "13 | Tabela DDLDEPENDENCY")
dbutils.widgets.text("catalog_target", "platform_dev",       "14 | Catalog destino")
dbutils.widgets.text("schema_target",  "sap_s4_nc2_replica", "15 | Schema destino")

print("🎛️  Widgets criados.")

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
from pathlib import Path

_aqui = Path.cwd()
for _cand in [_aqui, *_aqui.parents]:
    if (_cand / "src" / "rosetta" / "__init__.py").exists():
        RAIZ = _cand
        break
else:
    raise FileNotFoundError(f"Raiz do repositório não encontrada a partir de {_aqui}")

if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

import rosetta
from rosetta import Config, Contexto, certeza_real
from rosetta import localizador

CFG = Config.de_widgets(dbutils)
print(f"📦 rosetta {rosetta.__version__}")
print(CFG.resumo())

## 2. 🔎 Pré-seleção estrutural (Motor 01)

Classifica cada fonte em três eixos: completude (não foi cortado pelo RFC 32K), tipo
(vira `CREATE VIEW` ou não) e fecho de dependências (nada truncado acima na cadeia).

In [0]:
INV = localizador.rodar(spark, CFG)
print()
print(INV.resumo())

In [0]:
display(spark.createDataFrame(
    INV.pdf["motivo"].value_counts().rename_axis("motivo").reset_index(name="qtd")
))

## 3. 🏆 Certeza real

Roda `parse_cds` + `gerar_sql` em cada apta. Só entra em `GARANTIDAS` quem gerar SQL com
**zero avisos** — sem `$session`, sem função ABAP de data/timestamp, sem `abap.*` pendente,
sem cardinalidade de association arriscada, sem `$projection` órfão, sem `with parameters`.

In [0]:
CTX = Contexto(spark, CFG, raiz_ddl=RAIZ / "ddl",
               truncados_conhecidos=INV.truncadas)

CERTEZA = certeza_real(CTX, INV.aptas)
print()
print(CERTEZA.resumo())

In [0]:
# As garantidas, da mais simples para a mais complexa — bons candidatos para
# validar o tradutor manualmente no Databricks antes de escalar.
_g = CERTEZA.pdf[CERTEZA.pdf["ddlname"].isin(CERTEZA.garantidas)]
display(spark.createDataFrame(
    _g[["ddlname", "entidade", "tipo", "n_campos"]].sort_values("n_campos").reset_index(drop=True)
))

## 4. 🔬 Onde estão os avisos

Agrupa por mensagem para mostrar o que mais bloqueia views hoje — é o que diz qual regra
de tradução vale a pena implementar em seguida.

In [0]:
from collections import Counter

_cont = Counter()
for txt in CERTEZA.pdf.loc[CERTEZA.pdf["n_avisos"] > 0, "avisos"]:
    for a in txt.split(" | "):
        if a:
            _cont[a.split(" — ")[0]] += 1

print("=" * 78)
print("  🔬 AVISOS MAIS FREQUENTES (o que impede mais views de serem garantidas)")
print("=" * 78)
for msg, n in _cont.most_common(20):
    print(f"   {n:>7,}  {msg}")
print("=" * 78)

## 5. 📤 Exportar a lista de garantidas

Grava `ddl/_inventario/garantidas.csv` para o time consultar fora do notebook. É um
arquivo de texto no repositório — nada é escrito no catálogo.

In [0]:
PASTA_INV = RAIZ / "ddl" / "_inventario"
PASTA_INV.mkdir(parents=True, exist_ok=True)

CERTEZA.pdf.sort_values(["gerou_sql", "n_avisos", "ddlname"],
                        ascending=[False, True, True]).to_csv(
    PASTA_INV / "certeza_real.csv", index=False, encoding="utf-8")

(PASTA_INV / "garantidas.txt").write_text(
    "\n".join(sorted(CERTEZA.garantidas)) + "\n", encoding="utf-8")

print(f"💾 {PASTA_INV / 'certeza_real.csv'}")
print(f"💾 {PASTA_INV / 'garantidas.txt'}  ({len(CERTEZA.garantidas):,} views)")
print("\n👉 Pegue um nome de garantidas.txt e rode o notebook 01_gerar_view com ele.")